# Diagnose llcRental Accounting Books 

 

In [6]:
import os
from pathlib import Path
import json
import pandas as pd
from IPython.display import display, Markdown

In [2]:
### -- ExpRev Global Change

# import json



In [8]:
# llcExpRev Data
import pandas as pd
from IPython.display import display, Markdown

fnDict = dict(asset = Path.home()/ 'GDrive/Family/Assets/LLC-WBGroup/books/Accts/llcAsset_WBGroupLLC.json',
              er = Path.home()/ 'GDrive/Family/Assets/LLC-WBGroup/books/Accts/llcExpRev_WBGroupLLC.json')
    
def loadLedger(fn):
    jstr = fn.read_text()
    return json.loads(jstr)

def saveLedger(erList, fn):
    with open(fn,'w') as fio:
        json.dump(erList, fio)
    
def displayLedger(erList, **kwargs):
    for er in erList:
        amt = er['amt']
        pNm = er['propNm']
        aSub = er['acctSub']
        a = er['acct']
        l = er['Ledger']
    
        print(f"{pNm:13s} {amt:10.2f}: {er['aType']:7s}, {a} -> {l} ==acctSub:{aSub}")

def displayGroup(erList):
    df = pd.DataFrame(erList)
    return df.groupby(['acct', 'Ledger']).amt.sum()

def swapLedger(erList):
    for er in erList:
        amt = er['amt']
        pNm = er['propNm']
        aSub = er['acctSub']
        a = er['acct']
        l = er['Ledger']
        dc = er['aType']
        
        if er['acct'] != 'Acct.Cash.Bank' and er['Ledger'] == 'Acct.Cash.Bank':
            # swap ExpRev so Acct.Cash.banik
            er['acct'] = l
            er['Ledger'] = a
            er['aType'] = 'Credit' if dc == 'Debit' else 'Debit'
    
def cleanLedger(erOld):
    print(f"\n=========== Old ExpRev {len(erOld)} ==========")

    erList = []
    for er in erOld:
        # Clean transactions
        pNm = er['propNm']
        aSub = er['acctSub']
    
        a = er['acct']
        l = er['Ledger']
        dc = er['aType']
    

        # Ignore non RV_RV1
        if 'RV_RV1' != er['propNm']:                
            erList.append(er)
            continue
            
        # Ignore investment, purchase
        amt = er['amt']
        if amt == 177.00 :
            erList.append(er)
            #print(er)
            continue

        er['acct'] = 'Acct.Cash.Bank'
        er['Ledger'] = 'Acct.Fixed.Tangible.InConstruction'
        er['aType'] = 'Credit'
    
        print(f"{pNm:7s} {amt:10.2f}: {er['aType']:7s}, {a} -> {l} ==acctSub:{aSub}")
        erList.append(er)
    return erList

In [25]:
# Load llcExpRev

# customize this lambda to filter only transactins desired
testRec = lambda er : er['propNm'] == 'RV_RV1'

erList = loadLedger(fnDict['er'])

def displayLedger(erList, **kwargs):
    for er in erList:
        amt = er['amt']
        pNm = er['propNm']
        aSub = er['acctSub']
        a = er['acct']
        l = er['Ledger']

        func = kwargs.get('func', None)

        # If func and func is True - display, else ignore
        if func is not None:
            if not func(er) : continue
    
        print(f"{pNm:13s} {amt:10.2f}: {er['aType']:7s}, {a} -> {l} ==acctSub:{aSub} >>> tid:{er['tID']}")

displayLedger(erList, func=testRec)

RV_RV1             27.04: Credit , Acct.Cash.Bank -> Acct.Fixed.Tangible.InConstruction ==acctSub:None >>> tid:2025.10.07_C27.04
RV_RV1             27.04: Credit , Acct.Cash.Bank -> Acct.Fixed.Tangible.InConstruction ==acctSub:None >>> tid:2025.10.07_C27.04_2
RV_RV1             31.86: Credit , Acct.Cash.Bank -> Acct.Fixed.Tangible.InConstruction ==acctSub:Exp Other >>> tid:2025.10.07_C31.86
RV_RV1             34.63: Credit , Acct.Cash.Bank -> Acct.Fixed.Tangible.InConstruction ==acctSub:Exp Other >>> tid:2025.10.08_C34.63
RV_RV1             37.87: Credit , Acct.Cash.Bank -> Acct.Fixed.Tangible.InConstruction ==acctSub:Const >>> tid:2025.10.09_C37.87
RV_RV1            108.32: Credit , Acct.Cash.Bank -> Acct.Fixed.Tangible.InConstruction ==acctSub:Exp Other >>> tid:2025.10.14_C108.32
RV_RV1             19.47: Credit , Acct.Cash.Bank -> Acct.Fixed.Tangible.InConstruction ==acctSub:Const >>> tid:2025.10.14_C19.47
RV_RV1             30.71: Credit , Acct.Cash.Bank -> Acct.Fixed.Tangible.InCo

In [22]:
displayLedger(loadLedger(fnDict['er']), func=testRec)

H_805HighMesa      14.06: Credit , Acct.Cash.Bank -> Acct.Exp.Repair ==acctSub:Repair >>> tid:2025.10.07_C14.06
H_805HighMesa      14.06: Debit  , Acct.Cash.Bank -> Acct.Exp.Other ==acctSub:Repair >>> tid:2025.10.09_D14.06
H_805HighMesa      14.06: Credit , Acct.Cash.Bank -> Acct.Exp.Other ==acctSub:Repair >>> tid:2025.10.09_D14.06_2


## Diagnose GL

- load GL
- proof: trace every GL transaction back to llc object


In [32]:
# stmtGL data

from ledger import setup_paths as _sp
from ledger.LLC import LLC
from ledger.stmtGL import stmtGL

_sp.load_config('WBGroupLLC', 2025)
llc = LLC('WBGroupLLC')
gl = stmtGL(llc)
glList = gl.load()
len(glList)

#pd.DataFrame([t for t in glList if abs(t['amt']) == 14.06])
pd.DataFrame([t for t in glList if t['propNm'] == 'RV_RV1'])

[setup_paths] Loaded 'WBGroupLLC/2025' from /Users/frankrojas/.llcRentalTracker/config.json → bus_repo=/Users/frankrojas/Library/CloudStorage/GoogleDrive-frankr6591@gmail.com/My Drive/Family/Assets/LLC-WBGroup


,Status,dt,acctType,acct,aType,amt,desc,acctSub,propNm,propOwners,refDB,tID,srcTID,_lineNo,_rowNm
0,,2025.08.30,Equity,Acct.Equity.Owner.Capital.Funds,Credit,422.97,Payback Member from Owner's Capital Fund,Const,RV_RV1,o20250801_1:100%,llcPayable,2025.08.30_-422.97,2025.08.30_422.97,74,2025.08.30_-422.97
1,,2025.08.30,Asset,Acct.Fixed.Tangible.InConstruction,Debit,422.97,Payback Member from Owner's Capital Fund,Const,RV_RV1,o20250801_1:100%,llcPayable,2025.08.30_422.97,2025.08.30_422.97,75,2025.08.30_422.97
2,,2025.09.30,Equity,Acct.Equity.Owner.Capital.Funds,Credit,290.95,Payback Member from Owner's Capital Fund - Sept,Const,RV_RV1,o20250801_1:100%,llcPayable,2025.09.30_-290.95,l2025.09.30_290.95-AP1,84,2025.09.30_-290.95
3,,2025.09.30,Asset,Acct.Fixed.Tangible.InConstruction,Debit,290.95,Payback Member from Owner's Capital Fund - Sept,Const,RV_RV1,o20250801_1:100%,llcPayable,2025.09.30_290.95,l2025.09.30_290.95-AP1,85,2025.09.30_290.95
4,,2025.10.07,Asset,Acct.Cash.Bank,Credit,27.04,NoRcpt Approved Purchase: LOWE'S #159 SAN MARCOS,,RV_RV1,,llcBank,2025.10.07_-27.04,2025.10.07_C27.04,92,2025.10.07_-27.04
5,,2025.10.07,Asset,Acct.Cash.Bank,Credit,27.04,NoRcpt Approved Purchase: LOWE'S #159 SAN MARCOS,,RV_RV1,,llcBank,2025.10.07_-27.04_001,2025.10.07_C27.04_2,93,2025.10.07_-27.04_001
6,,2025.10.07,Asset,Acct.Fixed.Tangible.InConstruction,Debit,27.04,NoRcpt Approved Purchase: LOWE'S #159 SAN MARCOS,,RV_RV1,,llcBank,2025.10.07_27.04,2025.10.07_C27.04,94,2025.10.07_27.04
7,,2025.10.07,Asset,Acct.Fixed.Tangible.InConstruction,Debit,27.04,NoRcpt Approved Purchase: LOWE'S #159 SAN MARCOS,,RV_RV1,,llcBank,2025.10.07_27.04_001,2025.10.07_C27.04_2,95,2025.10.07_27.04_001
8,,2025.10.07,Asset,Acct.Cash.Bank,Credit,31.86,NoRcpt Approved Purchase: LOWES #00907* 866-48...,Exp Other,RV_RV1,,llcBank,2025.10.07_-31.86,2025.10.07_C31.86,96,2025.10.07_-31.86
9,,2025.10.07,Asset,Acct.Fixed.Tangible.InConstruction,Debit,31.86,NoRcpt Approved Purchase: LOWES #00907* 866-48...,Exp Other,RV_RV1,,llcBank,2025.10.07_31.86,2025.10.07_C31.86,97,2025.10.07_31.86
